# THEREDA + extra GEMS3 records → ThermoHub

Imports GEMS3 backup records (`IComp` → elements, `DComp` → substances) sitting in
`data-in/add-gems-records/` **on top of** the graph `import-THEREDA-json-data.ipynb` already
built, and wires them into the existing THEREDA thermodataset and phases:

| GEMS3 record | ThermoHub vertex | linked to |
|---|---|---|
| `IComp.*.json` | `VertexElement` | the THEREDA thermodataset, via an `EdgeBasis` |
| `DComp.*.json`, key `a…` (aqueous) | `VertexSubstance` | the `aq` Pitzer phase (`M_AQPITZERHMW`), via an `EdgeComponent` |
| `DComp.*.json`, key `g…`/`f…` (gas/fluid) | `VertexSubstance` | the `g` Peng-Robinson-78 phase (`M_PR78FLUID`), via an `EdgeComponent` |

**Run order:** `import-THEREDA-json-data.ipynb` → *this notebook* → 
`export-THEREDA-GEMS3-data.ipynb`. All three must use the same database name.

The record import itself is fully declarative, driven by two new `scripts-in` format files
(`Element.gems3.FormatStructDataFile.json`, `Substance.gems3.FormatStructDataFile.json`) via the
generic `ReacDCImport` generator — the same way `databases/MINES/import-mines-gems3-json-data.ipynb`
reads Mines23's GEMS3 json. The graph wiring at the end is *not* declarative, and deliberately so:
the `aq`/`g` phase and the thermodataset vertices already exist, and re-importing them through a
`records_and_links` row would **replace** those documents (losing THEREDA's own substance list,
mixing model and T/P limits) instead of extending them. `gems3_add_records.ThermoHubGraph` adds
the edges and appends to `properties.substances` in place.

## 1. Collect the GEMS3 drops into import blocks

`data-in/add-gems-records/` holds GEMS3 backup files under their natural names
(`IComp.<tag>.json`, `DComp.<tag>.json`); drop in as many as you like. `collect_gems3_blocks()`
merges all `IComp.*` into `add-gems.elements.json` and all `DComp.*` into `add-gems.substances.json`,
because `ReacDCImportData::splitToBlocks()` locates each block's file by replacing the
next-to-last dot-separated part of the input file name with the block name.

It also reports which phase each DComp belongs to, read from the first field of its GEMS3 key
(the aggregate-state code, `a`/`g`/`f`/…).

In [1]:
import sys
sys.path.append("../..")
from gems3_add_records import collect_gems3_blocks, ThermoHubGraph

DB_NAME = "ORD_THEREDA_R17"      # same database as the import/export notebooks
THERMODATASET = "THEREDA2026"            # rcsetSymbol used by import-THEREDA-json-data.ipynb

blocks = collect_gems3_blocks("data-in/add-gems-records")
for path, count in blocks["sources"]:
    print(f"{count:4d} record(s) from {path}")
print("elements:  ", blocks["elements"])
for substance in blocks["substances"]:
    print("substance: ", substance)

   1 record(s) from data-in/add-gems-records/IComp.N_atm.json
   2 record(s) from data-in/add-gems-records/DComp.N_atm.json
elements:   ['N_atm']
substance:  {'symbol': 'N2(aq_atm)', 'state': 'a', 'phase': 'aq'}
substance:  {'symbol': 'N2(g_atm)', 'state': 'f', 'phase': 'g'}


Anything reported with `'phase': None` has an aggregate-state code this notebook has no phase
for (a solid, say). Its `DComp` still gets imported as a substance below, but nothing links it,
so it will not appear in the GEMS3 export — add it to a phase by hand, or extend
`AGGREGATE_STATE_PHASE` in `gems3_add_records.py`.

## 2. Connect to the already-populated database

In [2]:
import thermomatch as match

# This notebook extends an existing graph - a wrong database name must fail loudly rather than
# quietly create an empty database with two orphan records in it.
match.ThermoImpexGenerator.create_local_database_if_not_existent = False

# Set path to `schemas` and `lua` not from the configuration file
match.ThermoImpexGenerator.setResourcesDirectory("../../Resources")

# Same THEREDA redox/formula rules the import and export notebooks load; formula charge-fixing
# runs on every VertexSubstance created below.
match.loadGems3ExportRulesFile("scripts-out/gems3-export-rules.json")

generator = match.ThermoImpexGenerator("http://localhost:8529", "root", "", DB_NAME,
                                       overwrite_existing=True,
                                       formats_from=match.ThermoImpexGenerator.File,
                                       split_to_blocks=False)

[2026-09-03 10:50:00.418] [jsonio17] [info] Home directory is /home/dmiron
[thermomatch] [info] jsonio17::ioSettings() is reading settings from "../../Resources/ThermoMatch-config.json" (resolved to "/home/dmiron/git/hub/thermoimpex-jupyter/databases/THEREDA/../../Resources/ThermoMatch-config.json", cwd "/home/dmiron/git/hub/thermoimpex-jupyter/databases/THEREDA")
[thermomatch] [info] jsonio17::ioSettings() common.ResourcesDirectory = "../../Resources"
[thermomatch] [info] Loaded Gems3Export rules from "scripts-out/gems3-export-rules.json" (RedoxElements=29 FormulaOverrides=1 SymbolShortenRules=15 SDrefAuthorKeyOverrides=3 ModeCodeRules=7)
[jsonarango] [info] You are connected to: arango 3.9.12


## 3. Import the records

Elements first: `fixImportedSubstanceFormulas` parses each new substance's formula against the
elements it finds in the database, so a `DComp` whose formula uses a newly added `IComp`
(`N_atm2` uses `N_atm`) needs that element to already be there.

Both rows are plain `"records"` — no `"records_and_links"`, no `"group"` — so `generateEdges()`
does nothing and no reaction set or thermodataset vertex is created. Step 4 does all the linking.

In [3]:
description = [
    match.ImpexRowDescription("elements", "VertexElement", "records", "FormatStructDataFile",
                              "scripts-in/Element.gems3.FormatStructDataFile.json"),   # GEMS3 IComp records
    match.ImpexRowDescription("substances", "VertexSubstance", "records", "FormatStructDataFile",
                              "scripts-in/Substance.gems3.FormatStructDataFile.json"), # GEMS3 DComp records
]

generator.runImport("ReacDCImport", description, blocks["input_file"], THERMODATASET)

[thermomatch] [info]  0% Preparing files ...
[thermomatch] [info]  File name for block data-in/add-gems-records/add-gems.elements.json
[thermomatch] [info]  File name for block data-in/add-gems-records/add-gems.substances.json
[thermomatch] [info]  10% Importing data ...
[thermomatch] [info]  10% Importing elements VertexElement ...
[jsonio17] [info] VertexElement loading collection: 1371, loading query: 3223
[thermomatch] [info]  40% Importing substances VertexSubstance ...
[jsonio17] [info] VertexSubstance loading collection: 982, loading query: 110334
[thermomatch] [info]  80% Load database to compare ...
[thermomatch] [info]  90% Generate Edges ...
[thermomatch] [info]  100% Ready to test


## 4. Link the new records into the existing thermodataset and phases

`ExportToGems3` selects what to export by traversing `basis`/`belongs`/`involves`/`prodreac`/
`product`/`component` edges INBOUND from the thermodataset (`select_linked()`), so these two
steps are what actually makes the new records reach `IComp.backup.json`/`DComp.backup.json`:

* elements → `basis` edge to the thermodataset;
* substances → `component` edge to their phase, *and* their symbol appended to that phase's
  `properties.substances` (which is what `get_PhasePitzer()` reads to build the GEMS3 phase's
  DC list — the edge alone would export the substance but leave the phase record unchanged).

Both are idempotent: re-running the notebook does not duplicate edges or list entries.

In [4]:
graph = ThermoHubGraph(database=DB_NAME)

for entry in graph.add_elements_to_thermodataset(blocks["elements"], THERMODATASET):
    print("basis edge", entry)

by_phase = {}
for substance in blocks["substances"]:
    by_phase.setdefault(substance["phase"], []).append(substance["symbol"])

for phase_symbol, symbols in sorted(by_phase.items(), key=lambda item: str(item[0])):
    if phase_symbol is None:
        print("NOT LINKED (no phase for this aggregate state):", symbols)
        continue
    result = graph.add_substances_to_phase(symbols, phase_symbol)
    for entry in result["components"]:
        print(f"component edge -> phase {phase_symbol!r}:", entry)
    print(f"phase {phase_symbol!r} now has {len(result['substances'])} substances")

basis edge {'element': 'N_atm', 'edge': 'basis/32462783', 'created': True}
component edge -> phase 'aq': {'substance': 'N2(aq_atm)', 'edge': 'component/32462792', 'created': True}
phase 'aq' now has 231 substances
component edge -> phase 'g': {'substance': 'N2(g_atm)', 'edge': 'component/32462803', 'created': True}
phase 'g' now has 5 substances


## What this does and does not change

Verified end-to-end against a copy of the THEREDA database (import, then `ExportAllJson`, then a
record-by-record diff against the same export without this notebook):

* `IComp.backup.json` and `DComp.backup.json` gain exactly the added records, round-tripped
  faithfully — same `G0d`/`H0d`/`S0d`/`Cp0d`/`V0d`/`aiCpT`/`TCint`/`TPcMod`/`DCset` values.
* the `aq` phase record in `Phase.backup.json` grows by each added aqueous species
  (`Ph_Nc`, `pDC_v_`, `PhDCC`, `PhDCS`, and the re-indexed Pitzer interaction matrix `ipxT`).
* nothing else moves: `ReacDC.backup.json`, `SDref.backup.json` and every other phase are
  byte-identical apart from `Cdate`/`rCdat`.

Caveats worth knowing:

* **The `g` phase is still not exported at all.** `ExportToGems3::splitPureAndMixedPhaseIds()`
  skips `symbol == "g"` outright, so a gas added here reaches `DComp.backup.json` (through the
  `component` → `g` → `belongs` traversal) but no `g` phase record appears in `Phase.backup.json`.
  That is pre-existing behaviour, not something this notebook introduces.
* **An added aqueous species has no Pitzer interaction parameters.** It joins the `aq` phase as
  a species the mixing model has no `beta`/`theta`/`psi` terms for.
* **GEMS3 `dSDval` labels are not preserved.** `dSDref`/`dSDval` are imported as the substance's
  flat `properties.datasources` + `properties.comment` (the record-level convention
  `collect_datasource_refs()` supports for non-THEREDA sources), so the export re-emits the
  record's comment as the label for every reference instead of the source's per-property
  `DfGm_Ref`/`Sm_Ref`/… labels. References with no matching `VertexDataSource` are reported by
  the export as *"SDref data source not found"* and pass through as raw shortnames.
* **The exported DC symbol is shortened.** THEREDA's `SymbolShortenRules` in
  `scripts-out/gems3-export-rules.json` apply to these records too, so e.g. `N2(g_atm)` is
  exported under the GEMS3 key `N2g_atm`.
* The added records carry `sourcetdb = {"12": "THEREDA"}` (set in both `scripts-in` format
  files) so they key and match like the rest of the dataset. Change it there — `{"1111":
  "OTHER"}` is the neutral code — if you would rather keep them distinguishable.